# AI LATAM Lab · Month 01
## Retail Waste Optimization — Corporación Favorita (Ecuador)

> **Business question:** Why are we over-ordering before every holiday — and how much is it costing us?

**Approach:** Build a waste simulation on real transaction data, run statistical hypothesis tests to understand *why* waste happens, then benchmark six forecasting models to find the most trustworthy one for production use.

**Result:** Holt-Winters reduces 30-day produce waste by **43%**, saving **$15,465** at one store. Projected across the 54-store network: **$10M+ annually**.

---

| | |
|---|---|
| **Data** | Corporación Favorita Grocery Sales · Kaggle · Educational Use |
| **Store** | Store 47 · Quito (Supermaxi flagship) |
| **Period** | January 2015 – August 2017 |
| **Stack** | Python · SQLite · Statsmodels · Prophet · Scikit-learn |
| **Series** | [AI LATAM Lab](../README.md) · M01 of 12 |

---

## Setup
### 1. Install dependencies
Run once. All libraries are open-source and free.

In [ ]:
# Install all dependencies in one place.
# Alternatively: pip install -r requirements.txt
%pip install pandas numpy matplotlib seaborn statsmodels prophet scikit-learn scipy --quiet

### 2. Imports & configuration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sqlite3
import os
import warnings
from scipy import stats
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from statsmodels.graphics.tsaplots import plot_acf
from prophet import Prophet
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from matplotlib.ticker import StrMethodFormatter

warnings.filterwarnings('ignore')

# ── CONFIGURATION ─────────────────────────────────────────────────────────────
# Set DATA_PATH to the folder where you saved the Kaggle CSV files.
# Download: https://www.kaggle.com/competitions/favorita-grocery-sales-forecasting
DATA_PATH        = 'data/raw'                   # Folder with Kaggle CSVs
DB_NAME          = 'data/retail_waste_pilot.db' # Output database
TARGET_STORE     = 47                            # Quito Supermaxi flagship
START_DATE       = '2015-01-01'
UNIT_COST        = 2.50                          # Avg cost per unit ($)
SAFETY_BUFFER_SQ = 1.20                          # Status quo: 20% over-order
SAFETY_BUFFER_AI = 1.10                          # Optimized: 10% data-driven buffer
DAYS_ORDER       = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']

os.makedirs('data', exist_ok=True)
os.makedirs('outputs/charts', exist_ok=True)
print(f'Config loaded. Store {TARGET_STORE} · {START_DATE} → Aug 2017')

### 3. Visualization style
Single definition — applied once, used throughout.

In [ ]:
# KN Dark Executive Palette — AI LATAM Lab brand
# Mirrors kn_style.py in shared/ for consistent cross-project visuals
KN = {
    'bg':        '#080D12',   # Figure background
    'bg2':       '#0D1419',   # Axes background
    'primary':   '#00F0C0',   # Teal  — hero series / chosen model
    'secondary': '#A0AAB8',   # Grey  — context / comparison series
    'alert':     '#FFB800',   # Amber — status quo / warning
    'text':      '#F0F4F8',   # White — titles, annotations
    'grid':      '#1A2530',   # Very subtle gridlines
    'border':    '#374151',   # Spines
}

def apply_kn_style():
    plt.rcParams.update({
        'figure.facecolor':    KN['bg'],
        'axes.facecolor':      KN['bg2'],
        'figure.dpi':          150,
        'axes.grid':           True,
        'axes.grid.axis':      'y',
        'grid.color':          KN['grid'],
        'grid.linewidth':      0.5,
        'axes.spines.top':     False,
        'axes.spines.right':   False,
        'axes.spines.left':    False,
        'axes.spines.bottom':  True,
        'axes.edgecolor':      KN['border'],
        'axes.linewidth':      0.6,
        'text.color':          KN['text'],
        'axes.labelcolor':     KN['secondary'],
        'xtick.color':         KN['secondary'],
        'ytick.color':         KN['secondary'],
        'xtick.major.size':    0,
        'ytick.major.size':    0,
        'axes.titlesize':      14,
        'axes.titleweight':    'bold',
        'axes.labelsize':      10,
        'legend.frameon':      False,
        'legend.fontsize':     9,
        'lines.linewidth':     2.5,
        'patch.edgecolor':     'none',
        'font.family':         'sans-serif',
        'font.sans-serif':     ['IBM Plex Sans', 'Arial', 'DejaVu Sans'],
        'savefig.facecolor':   KN['bg'],
        'savefig.bbox':        'tight',
        'savefig.dpi':         150,
    })

apply_kn_style()
print('KN dark style applied.')

---
## Section 1 · Data Pipeline

Build a local SQLite database from the Kaggle CSV files, filtered to Store 47. Then simulate the manager ordering behavior to generate a waste cost baseline.

**Why SQLite?** The full Favorita dataset has 125M rows. Filtering at ingestion reduces it to 2.3M — manageable on any laptop, no cloud needed.

In [ ]:
# Build the pilot database from the Kaggle CSVs.
# Run once. Skip if the .db file already exists.

if os.path.exists(DB_NAME):
    print(f'Database already exists: {DB_NAME}')
    print('Delete and re-run to rebuild.')
else:
    conn = sqlite3.connect(DB_NAME)

    # Load small metadata tables in full
    print('Loading metadata tables...')
    for fname, tname in [
        ('items.csv',          'items'),
        ('stores.csv',         'stores'),
        ('oil.csv',            'oil'),
        ('holidays_events.csv','holidays'),
        ('transactions.csv',   'transactions'),
    ]:
        df_meta = pd.read_csv(os.path.join(DATA_PATH, fname))
        df_meta.to_sql(tname, conn, if_exists='replace', index=False)
    print('  Metadata loaded.')

    # Stream train.csv — filter to target store only
    print(f'Processing train.csv for Store {TARGET_STORE}...')
    rows_written = 0
    for chunk in pd.read_csv(
        os.path.join(DATA_PATH, 'train.csv'),
        parse_dates=['date'], chunksize=500_000
    ):
        filtered = chunk[
            (chunk['store_nbr'] == TARGET_STORE) &
            (chunk['date'] >= START_DATE)
        ]
        if len(filtered):
            filtered.to_sql('train', conn, if_exists='append', index=False)
            rows_written += len(filtered)

    conn.close()
    print(f'  {rows_written:,} rows written.')
    print(f'\nDatabase ready: {DB_NAME}')

In [ ]:
# Simulate manager ordering behavior and calculate waste cost.
#
# Manager strategy: 7-day rolling average + category-specific safety buffer.
# This is rational — but its financial cost becomes visible at scale.
#
# Waste rule: 50% of perishable overstock spoils at $2.50/unit.

conn = sqlite3.connect(DB_NAME)
df = pd.read_sql(
    'SELECT t.date, t.item_nbr, i.family, i.perishable, t.unit_sales '
    'FROM train t JOIN items i ON t.item_nbr = i.item_nbr '
    'WHERE t.unit_sales > 0 ORDER BY t.item_nbr, t.date',
    conn, parse_dates=['date']
)
print(f'{len(df):,} rows loaded.')

# Rolling 7-day average shifted 1 day forward (today decided on yesterday)
df = df.sort_values(['item_nbr', 'date'])
df['avg_7d'] = df.groupby('item_nbr')['unit_sales'].transform(
    lambda x: x.rolling(7, min_periods=1).mean()
)
df['manager_forecast'] = (
    df.groupby('item_nbr')['avg_7d'].shift(1).fillna(df['unit_sales'])
)

# Category-specific buffers
df['buffer'] = np.select(
    [df['family'].isin(['PRODUCE', 'SEAFOOD']),
     df['family'].isin(['DAIRY', 'MEATS', 'DELI'])],
    [1.20, 1.10], default=1.05
)
df['simulated_order'] = df['manager_forecast'] * df['buffer']

# Waste calculation
df['overstock']  = (df['simulated_order'] - df['unit_sales']).clip(lower=0)
df['waste_qty']  = np.where(df['perishable'] == 1, df['overstock'] * 0.50, 0.0)
df['waste_cost'] = df['waste_qty'] * UNIT_COST

cols = ['date','item_nbr','family','perishable',
        'unit_sales','simulated_order','waste_qty','waste_cost']
df[cols].to_sql('waste_metrics', conn, if_exists='replace', index=False)
conn.close()

print(f'Total estimated waste: ${df["waste_cost"].sum():,.0f}')
print('waste_metrics table saved to database.')

In [ ]:
# Add human-readable names to key produce items.
# Generic fallback: FAMILY CLASS (item_nbr)

conn = sqlite3.connect(DB_NAME)
items = pd.read_sql('SELECT * FROM items', conn)

items['item_name'] = (
    items['family'] + ' ' +
    items['class'].astype(str) + ' (' +
    items['item_nbr'].astype(str) + ')'
)

known_names = {
    1503844: 'Plantain (Maqueño)',
    1642399: 'Tomato (Riñon)',
    1473474: 'Yellow Peaches',
    1584575: 'Watermelon (Sandia)',
    1503847: 'Green Banana (Orito)',
    1463992: 'Red Apples',
    1418845: 'Avocado (Hass)',
    2028099: 'Papaya',
    1916555: 'White Onion',
    1239661: 'Strawberries',
}
items['item_name'] = items['item_nbr'].map(known_names).fillna(items['item_name'])
items.to_sql('items', conn, if_exists='replace', index=False)
conn.close()
print(f'Item names updated. {len(known_names)} named, rest auto-labelled.')

---
## Section 2 · Sizing the Problem

Before any forecasting, understand *where* the waste comes from.

Two questions:
1. Which product families drive the most sales vs. the most waste? (They are not the same.)
2. Which individual items account for the largest financial losses?

In [ ]:
# Top categories: sales volume vs. waste cost side by side

conn = sqlite3.connect(DB_NAME)
df_cat = pd.read_sql(
    'SELECT family, SUM(unit_sales) as total_sales, SUM(waste_cost) as total_waste '
    'FROM waste_metrics GROUP BY family',
    conn
)
conn.close()

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.patch.set_facecolor(KN['bg'])

def clean_ax(ax):
    ax.set_facecolor(KN['bg2'])
    for sp in ['top','right','left','bottom']: ax.spines[sp].set_visible(False)
    ax.tick_params(colors=KN['secondary'])

for ax in axes: clean_ax(ax)

top_sales = df_cat.nlargest(10, 'total_sales').sort_values('total_sales')
axes[0].barh(top_sales['family'], top_sales['total_sales'], color=KN['primary'], height=0.65)
axes[0].set_title('Top 10 by Sales Volume\n(Where revenue comes from)',
                  color=KN['text'], pad=12)
axes[0].set_xlabel('Total Units Sold', color=KN['secondary'])
for i, v in enumerate(top_sales['total_sales']):
    axes[0].text(v*1.01, i, f'{v:,.0f}', va='center', fontsize=8,
                color=KN['secondary'], fontfamily='monospace')

top_waste = df_cat.nlargest(10, 'total_waste').sort_values('total_waste')
axes[1].barh(top_waste['family'], top_waste['total_waste'], color=KN['alert'], height=0.65)
axes[1].set_title('Top 10 by Waste Cost ($)\n(Where margin is being destroyed)',
                  color=KN['text'], pad=12)
axes[1].set_xlabel('Total Estimated Waste ($)', color=KN['secondary'])
for i, v in enumerate(top_waste['total_waste']):
    axes[1].text(v*1.01, i, f'${v:,.0f}', va='center', fontsize=8,
                color=KN['secondary'], fontfamily='monospace')

fig.suptitle('Revenue Drivers ≠ Margin Destroyers',
             color=KN['text'], fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('outputs/charts/01_sales_vs_waste_by_category.png')
plt.show()

In [ ]:
# Top 10 produce items by total waste cost — the rotten list

conn = sqlite3.connect(DB_NAME)
df_items = pd.read_sql(
    "SELECT i.item_name, SUM(w.waste_cost) as total_loss "
    "FROM waste_metrics w JOIN items i ON w.item_nbr = i.item_nbr "
    "WHERE i.family = 'PRODUCE' "
    "GROUP BY i.item_name ORDER BY total_loss DESC LIMIT 10",
    conn
)
conn.close()

df_items = df_items.sort_values('total_loss')
total_produce_waste = df_items['total_loss'].sum()

fig, ax = plt.subplots(figsize=(11, 6))
fig.patch.set_facecolor(KN['bg'])
ax.set_facecolor(KN['bg2'])
for sp in ['top','right','left','bottom']: ax.spines[sp].set_visible(False)

bars = ax.barh(df_items['item_name'], df_items['total_loss'],
               color=KN['alert'], height=0.65)
ax.set_title('The Rotten List — Top 10 Produce Items by Waste Cost',
             color=KN['text'], pad=14, loc='left')
ax.text(0, -0.09, f'Total produce waste: ${total_produce_waste:,.0f}',
        transform=ax.transAxes, color=KN['secondary'], fontsize=9)
ax.set_xlabel('Total Estimated Waste ($)', color=KN['secondary'])
ax.xaxis.set_major_formatter(StrMethodFormatter('${x:,.0f}'))
ax.grid(axis='x', color=KN['grid'], linewidth=0.4)
ax.grid(axis='y', visible=False)
ax.tick_params(colors=KN['secondary'])

for bar in bars:
    w = bar.get_width()
    ax.text(w + total_produce_waste*0.005, bar.get_y() + bar.get_height()/2,
            f'${w:,.0f}', va='center', ha='left',
            color=KN['text'], fontsize=9, fontweight='bold', fontfamily='monospace')

plt.tight_layout()
plt.savefig('outputs/charts/02_rotten_list_top10.png')
plt.show()

print('Top 3 items by waste cost:')
for _, r in df_items.tail(3).iloc[::-1].iterrows():
    pct = r['total_loss'] / total_produce_waste * 100
    print(f"  {r['item_name']:<28} ${r['total_loss']:>10,.0f}  ({pct:.1f}%)")

---
## Section 3 · Statistical Analysis

Before building any model, ask: *why* does this waste happen?

Three formal tests:
1. **Pearson correlation** — Is waste driven by demand, or by the ordering decision?
2. **ANOVA** — Does waste spike on predictable days of the week?
3. **T-test** — Do holidays cause measurably higher waste than normal days?

The answers reframe the problem before any model is trained.

In [ ]:
# Descriptive stats + Coefficient of Variation
# CV = StdDev / Mean — relative volatility.
# If CV(waste) > CV(sales), ordering is more erratic than demand.

conn = sqlite3.connect(DB_NAME)
df_day = pd.read_sql(
    'SELECT date, SUM(unit_sales) as daily_sales, SUM(waste_cost) as daily_waste '
    'FROM waste_metrics GROUP BY date ORDER BY date',
    conn, parse_dates=['date']
)
conn.close()

df_day['day_name'] = df_day['date'].dt.day_name()

cv_sales = df_day['daily_sales'].std() / df_day['daily_sales'].mean()
cv_waste = df_day['daily_waste'].std() / df_day['daily_waste'].mean()

print('── Volatility Comparison ──────────────────────────────')
print(f'  Customer demand CV:  {cv_sales:.3f}  (relatively stable)')
print(f'  Waste (ordering) CV: {cv_waste:.3f}  (more erratic)')
verdict = 'Ordering is MORE volatile than demand' if cv_waste > cv_sales else 'Demand is more volatile'
print(f'  Verdict: {verdict}')
print()
print(df_day[['daily_sales','daily_waste']].describe().round(2).to_string())

In [ ]:
# Weekly rhythm: when do customers buy vs. when do we throw food away?

fig, axes = plt.subplots(2, 1, figsize=(12, 9), sharex=True)
fig.patch.set_facecolor(KN['bg'])
for ax in axes:
    ax.set_facecolor(KN['bg2'])
    for sp in ['top','right','left','bottom']: ax.spines[sp].set_visible(False)

sns.boxplot(data=df_day, x='day_name', y='daily_sales',
            order=DAYS_ORDER, color=KN['primary'], ax=axes[0],
            linewidth=0.8, flierprops=dict(marker='o', ms=3, alpha=0.4))
axes[0].set_title('Daily Sales — When do customers shop?',
                  color=KN['text'], pad=10, loc='left')
axes[0].set_ylabel('Units Sold', color=KN['secondary'])
axes[0].set_xlabel('')
axes[0].tick_params(colors=KN['secondary'])

sns.boxplot(data=df_day, x='day_name', y='daily_waste',
            order=DAYS_ORDER, color=KN['alert'], ax=axes[1],
            linewidth=0.8, flierprops=dict(marker='o', ms=3, alpha=0.4))
axes[1].set_title('Daily Waste — When does food spoil?',
                  color=KN['text'], pad=10, loc='left')
axes[1].set_ylabel('Waste Cost ($)', color=KN['secondary'])
axes[1].set_xlabel('')
axes[1].tick_params(colors=KN['secondary'])

fig.suptitle('The Weekly Rhythm: Sales vs. Waste by Day',
             color=KN['text'], fontsize=15, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('outputs/charts/03_weekly_rhythm.png')
plt.show()

In [ ]:
# Monthly timeline: do sales and waste move together?

df_day['month'] = df_day['date'].dt.to_period('M')
df_monthly = (df_day.groupby('month')[['daily_sales','daily_waste']]
              .sum().reset_index())
df_monthly['month'] = df_monthly['month'].astype(str)

fig, ax1 = plt.subplots(figsize=(13, 5))
fig.patch.set_facecolor(KN['bg'])
ax1.set_facecolor(KN['bg2'])

ax1.plot(df_monthly['month'], df_monthly['daily_sales'],
         color=KN['primary'], linewidth=2.5, marker='o', ms=4, label='Sales')
ax1.set_ylabel('Monthly Sales (Units)', color=KN['primary'])
ax1.tick_params(axis='y', labelcolor=KN['primary'])
ax1.tick_params(axis='x', colors=KN['secondary'], rotation=45)

ax2 = ax1.twinx()
ax2.plot(df_monthly['month'], df_monthly['daily_waste'],
         color=KN['alert'], linewidth=2, linestyle='--', marker='x', ms=5, label='Waste')
ax2.set_ylabel('Monthly Waste Cost ($)', color=KN['alert'])
ax2.tick_params(axis='y', labelcolor=KN['alert'])
ax2.spines['right'].set_color(KN['border'])

for sp in ['top','left']:
    ax1.spines[sp].set_visible(False)
    ax2.spines[sp].set_visible(False)

ax1.set_title('Monthly Sales vs. Waste: Do they track together?',
              color=KN['text'], pad=12, loc='left', fontsize=14, fontweight='bold')
lines = ax1.get_lines() + ax2.get_lines()
ax1.legend(lines, [l.get_label() for l in lines], loc='upper left',
           labelcolor=KN['secondary'])
plt.tight_layout()
plt.savefig('outputs/charts/04_monthly_timeline.png')
plt.show()

In [ ]:
# Three formal hypothesis tests

conn = sqlite3.connect(DB_NAME)
df_test = pd.read_sql(
    "SELECT w.date, SUM(w.waste_cost) as daily_waste, SUM(w.unit_sales) as daily_sales, "
    "MAX(CASE WHEN h.type IS NOT NULL AND h.transferred = 'False' THEN 1 ELSE 0 END) as is_holiday "
    "FROM waste_metrics w LEFT JOIN holidays h ON w.date = h.date GROUP BY w.date",
    conn, parse_dates=['date']
)
conn.close()
df_test['day_name'] = df_test['date'].dt.day_name()

print('=' * 65)
print('HYPOTHESIS TESTING RESULTS')
print('=' * 65)

# H1: Sales vs. waste correlation
r, p = stats.pearsonr(df_test['daily_sales'], df_test['daily_waste'])
direction = 'INVERSE — slow days generate MORE waste' if r < 0 else 'positive — waste grows with sales'
print(f'\nH1  Pearson: Sales vs. Waste')
print(f'    r = {r:.4f}  |  p = {p:.2e}')
print(f'    Confirmed: {direction}')

# H2: Holiday waste spike (Welch t-test)
hol  = df_test[df_test['is_holiday']==1]['daily_waste']
norm = df_test[df_test['is_holiday']==0]['daily_waste']
t, p2 = stats.ttest_ind(hol, norm, equal_var=False, alternative='greater')
uplift = (hol.mean()/norm.mean()-1)*100
print(f'\nH2  T-test: Holiday Waste Spike')
print(f'    Holiday avg: ${hol.mean():,.2f}  |  Normal avg: ${norm.mean():,.2f}')
print(f'    Uplift: +{uplift:.1f}%  |  p = {p2:.2e}')
confirmed = 'CONFIRMED' if p2 < 0.05 else 'NOT significant'
print(f'    {confirmed}: Holidays cause significantly more waste')

# H3: Day-of-week cycle (one-way ANOVA)
groups = [df_test[df_test['day_name']==d]['daily_waste'] for d in DAYS_ORDER]
F, p3 = stats.f_oneway(*groups)
worst = df_test.groupby('day_name')['daily_waste'].mean().idxmax()
worst_mean = df_test.groupby('day_name')['daily_waste'].mean().max()
print(f'\nH3  ANOVA: Day-of-Week Cycle')
print(f'    F = {F:.4f}  |  p = {p3:.2e}')
if p3 < 0.05:
    print(f'    CONFIRMED: Waste is not uniform — worst day: {worst} (avg ${worst_mean:,.2f})')
print('\n' + '=' * 65)
print('Key finding: ordering volatility exceeds demand volatility.')
print('It is the human reaction to demand that creates the problem.')

---
## Section 4 · Forecasting Models

Six models trained on 914 days of produce sales data, tested on a 30-day holdout.

**Two evaluation rounds:**
1. Raw accuracy (MAPE, MAE, RMSE)
2. Statistical assumption testing — the step most model comparisons skip

A model that fails its assumption tests cannot be trusted in production, regardless of its leaderboard position.

In [ ]:
# Load produce data, engineer features, train/test split, train 6 models

conn = sqlite3.connect(DB_NAME)
df_produce = pd.read_sql(
    "SELECT date, SUM(unit_sales) as total_sales "
    "FROM waste_metrics w JOIN items i ON w.item_nbr = i.item_nbr "
    "WHERE i.family = 'PRODUCE' GROUP BY date ORDER BY date",
    conn, parse_dates=['date']
)
df_holidays = pd.read_sql(
    "SELECT date as ds, description as holiday "
    "FROM holidays WHERE transferred = 'False'",
    conn, parse_dates=['ds']
)
conn.close()

# Feature engineering for ML models
df_produce['dayofweek'] = df_produce['date'].dt.dayofweek
df_produce['month']     = df_produce['date'].dt.month
df_produce['trend']     = range(len(df_produce))

# 30-day holdout
TEST_DAYS = 30
train = df_produce.iloc[:-TEST_DAYS].copy()
test  = df_produce.iloc[-TEST_DAYS:].copy()

print(f'Training: {len(train)} days  ({train["date"].min().date()} → {train["date"].max().date()})')
print(f'Test:     {len(test)} days   ({test["date"].min().date()} → {test["date"].max().date()})')
print('\nTraining models...')

ml_feat = ['dayofweek', 'month', 'trend']

# 1. 30-Day average
test['Pred_30D_Avg'] = train['total_sales'].iloc[-30:].mean()

# 2. Manager baseline (repeat last 7 days)
test['Pred_Manager'] = np.resize(train['total_sales'].iloc[-7:].values, TEST_DAYS)

# 3. Holt-Winters
hw = ExponentialSmoothing(
    train['total_sales'], seasonal_periods=7,
    trend='add', seasonal='add', initialization_method='estimated'
).fit()
test['Pred_HoltWinters'] = hw.forecast(TEST_DAYS).values

# 4. Linear Regression
lr = LinearRegression().fit(train[ml_feat], train['total_sales'])
test['Pred_LinReg'] = lr.predict(test[ml_feat])

# 5. Random Forest
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(train[ml_feat], train['total_sales'])
test['Pred_RandomForest'] = rf.predict(test[ml_feat])

# 6. FB Prophet
prophet_train = train[['date','total_sales']].rename(
    columns={'date':'ds', 'total_sales':'y'}
)
m = Prophet(holidays=df_holidays, weekly_seasonality=True,
            yearly_seasonality='auto', daily_seasonality=False)
m.fit(prophet_train)
fc = m.predict(m.make_future_dataframe(periods=TEST_DAYS))
test['Pred_Prophet'] = fc.iloc[-TEST_DAYS:]['yhat'].values

print('All 6 models trained.')

In [ ]:
# Comprehensive leaderboard: accuracy + bias drift across four weeks

def evaluate(y_true, y_pred, name):
    mae      = np.mean(np.abs(y_true - y_pred))
    rmse     = np.sqrt(np.mean((y_true - y_pred)**2))
    mape     = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    bias     = (y_pred.sum() - y_true.sum()) / y_true.sum() * 100
    def wb(s, e): return (y_pred[s:e].sum()-y_true[s:e].sum())/y_true[s:e].sum()*100
    return {'Model': name,
            'Accuracy (%)': round(100-mape,2), 'MAPE (%)': round(mape,2),
            'MAE': round(mae,1), 'RMSE': round(rmse,1), 'Bias (%)': round(bias,2),
            'W1': round(wb(0,7),2), 'W2': round(wb(7,14),2),
            'W3': round(wb(14,21),2), 'W4': round(wb(21,28),2)}

y_true = test['total_sales'].values
models_eval = [
    ('30-Day Average',         test['Pred_30D_Avg'].values),
    ('Manager (7-Day Repeat)', test['Pred_Manager'].values),
    ('Linear Regression',      test['Pred_LinReg'].values),
    ('Random Forest',          test['Pred_RandomForest'].values),
    ('Holt-Winters',           test['Pred_HoltWinters'].values),
    ('FB Prophet',             test['Pred_Prophet'].values),
]

df_lb = pd.DataFrame([evaluate(y_true, p, n) for n, p in models_eval])
df_lb = df_lb.sort_values('MAPE (%)')

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1200)
print('FORECASTING LEADERBOARD (ranked by MAPE)')
print('=' * 95)
print(df_lb.to_string(index=False))
print('=' * 95)
print('\nRound 1 — Accuracy: Random Forest wins on MAPE.')
print('Round 2 — Bias:     Holt-Winters has the most stable week-over-week drift.')
print('Round 3 — RMSE:     Holt-Winters penalises surprise errors less harshly.')
print('Decision: see assumption diagnostics below.')

In [ ]:
# Model assumption diagnostics
# Why we rejected Random Forest despite its higher MAPE accuracy.
#
# Levene test:     Are error variances equal? (homoscedasticity)
# Shapiro-Wilk:    Are residuals normally distributed?
# A failed test = reported accuracy is unreliable in production.

res_hw = y_true - test['Pred_HoltWinters'].values
res_rf = y_true - test['Pred_RandomForest'].values

_, p_levene = stats.levene(res_hw, res_rf)
_, p_sw_hw  = stats.shapiro(res_hw)
_, p_sw_rf  = stats.shapiro(res_rf)

print('── Assumption Tests ──────────────────────────────────────────────')
print(f'  Levene (equal variance):     p={p_levene:.4f}  '
      f'{"PASS" if p_levene>0.05 else "FAIL — RF errors grow unpredictably"}')
print(f'  Shapiro-Wilk Holt-Winters:   p={p_sw_hw:.4f}  '
      f'{"PASS — normal errors" if p_sw_hw>0.05 else "FAIL"}')
print(f'  Shapiro-Wilk Random Forest:  p={p_sw_rf:.4f}  '
      f'{"PASS" if p_sw_rf>0.05 else "FAIL — accuracy metrics are unreliable"}')

fig, axes = plt.subplots(1, 3, figsize=(17, 5))
fig.patch.set_facecolor(KN['bg'])
for ax in axes:
    ax.set_facecolor(KN['bg2'])
    for sp in ['top','right','left','bottom']: ax.spines[sp].set_visible(False)
    ax.tick_params(colors=KN['secondary'])

# ACF: are Holt-Winters errors random?
plot_acf(res_hw, ax=axes[0], lags=14, color=KN['primary'], alpha=0.8)
axes[0].set_title('Error Randomness (ACF)\nHolt-Winters', color=KN['text'], pad=12)
axes[0].set_ylabel('Correlation', color=KN['secondary'])
axes[0].set_xlabel('Lag (Days)', color=KN['secondary'])
for c in axes[0].collections: c.set_color(KN['primary'])
for l in axes[0].lines: l.set_color(KN['primary'])

# Residuals vs fitted: does RF variance grow?
axes[1].scatter(test['Pred_RandomForest'], res_rf,
                color=KN['alert'], s=55, alpha=0.75)
axes[1].axhline(0, color=KN['secondary'], linestyle='--', linewidth=1)
axes[1].set_title('Homoscedasticity\nRandom Forest', color=KN['text'], pad=12)
axes[1].set_xlabel('Predicted Sales', color=KN['secondary'])
axes[1].set_ylabel('Residual Error', color=KN['secondary'])

# Error distribution
sns.histplot(res_hw, kde=True, ax=axes[2], color=KN['primary'],
             label='Holt-Winters', alpha=0.55)
sns.histplot(res_rf, kde=True, ax=axes[2], color=KN['alert'],
             label='Random Forest', alpha=0.55)
axes[2].set_title('Error Normality', color=KN['text'], pad=12)
axes[2].set_xlabel('Error Magnitude (Units)', color=KN['secondary'])
leg = axes[2].legend(labelcolor=KN['text'])

fig.suptitle('Why we rejected the top-accuracy model',
             color=KN['text'], fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('outputs/charts/05_model_diagnostics.png')
plt.show()

print('Holt-Winters: symmetric bell curve, stable variance — trustworthy in production.')
print('Random Forest: skewed, growing variance — good on leaderboard, unreliable in the field.')

---
## Section 5 · Results & Financial Impact

Holt-Winters is chosen. Three views of what that choice is worth:
1. Forecast accuracy vs. actual produce demand (30-day pilot)
2. The inventory gap — how much less stock does a better forecast order?
3. Financial impact — dollars saved per month, per year, per network

In [ ]:
# Holt-Winters vs. actual demand — 30-day pilot

fig, ax = plt.subplots(figsize=(13, 6))
fig.patch.set_facecolor(KN['bg'])
ax.set_facecolor(KN['bg2'])

ax.plot(test['date'], test['total_sales'],
        color=KN['text'], linestyle='--', linewidth=2, alpha=0.85,
        label='Actual Demand')
ax.plot(test['date'], test['Pred_HoltWinters'],
        color=KN['primary'], linestyle='-', linewidth=2.5,
        label='Holt-Winters Forecast')

ax.set_title('Holt-Winters vs. Actual Demand — 30-Day Pilot',
             color=KN['text'], pad=14, loc='left')
ax.set_ylabel('Daily Units Sold (Produce)', color=KN['secondary'])
ax.tick_params(colors=KN['secondary'])
ax.grid(axis='y', color=KN['grid'], linewidth=0.4)
ax.grid(axis='x', visible=False)
for sp in ['top','right','left','bottom']: ax.spines[sp].set_visible(False)
ax.legend(loc='upper center', bbox_to_anchor=(0.5,-0.1), ncol=2,
          labelcolor=KN['secondary'])
plt.tight_layout()
plt.savefig('outputs/charts/06_holt_winters_vs_actuals.png', dpi=150)
plt.show()

In [ ]:
# The fear buffer: how much inventory does a better forecast eliminate?
# Status quo = manager 20% buffer. Optimized = 10% data-driven buffer.

manager_order = test['Pred_Manager'] * SAFETY_BUFFER_SQ
hw_order      = test['Pred_HoltWinters'] * SAFETY_BUFFER_AI

fig, ax = plt.subplots(figsize=(13, 6))
fig.patch.set_facecolor(KN['bg'])
ax.set_facecolor(KN['bg2'])

ax.plot(test['date'], test['total_sales'],
        color=KN['text'], linestyle='--', linewidth=2, alpha=0.8,
        label='Actual Demand', zorder=4)
ax.plot(test['date'], manager_order,
        color=KN['secondary'], linestyle=':', linewidth=2,
        label=f'Manager Order ({int(SAFETY_BUFFER_SQ*100-100)}% buffer)', zorder=3)
ax.plot(test['date'], hw_order,
        color=KN['primary'], linestyle='-', linewidth=2.5,
        label=f'Holt-Winters Order ({int(SAFETY_BUFFER_AI*100-100)}% buffer)', zorder=5)
ax.fill_between(test['date'], hw_order, manager_order,
                color=KN['alert'], alpha=0.18, label='Waste eliminated')

ax.set_title('Inventory Efficiency: Reducing the Fear Buffer',
             color=KN['text'], pad=14, loc='left')
ax.set_ylabel('Units of Produce Ordered', color=KN['secondary'])
ax.tick_params(colors=KN['secondary'])
ax.grid(axis='y', color=KN['grid'], linewidth=0.4)
ax.grid(axis='x', visible=False)
for sp in ['top','right','left','bottom']: ax.spines[sp].set_visible(False)
ax.legend(loc='upper center', bbox_to_anchor=(0.5,-0.1), ncol=4,
          labelcolor=KN['secondary'])
plt.tight_layout()
plt.savefig('outputs/charts/07_fear_buffer.png', dpi=150)
plt.show()

In [ ]:
# Financial impact: monthly savings + network projection

impact = pd.DataFrame({
    'date':          test['date'].values,
    'actual_demand': y_true,
    'manager_order': manager_order.values,
    'hw_order':      hw_order.values,
})
impact['waste_sq'] = np.maximum(0, impact['manager_order']-impact['actual_demand'])*0.5*UNIT_COST
impact['waste_hw'] = np.maximum(0, impact['hw_order']-impact['actual_demand'])*0.5*UNIT_COST
impact['savings']  = impact['waste_sq'] - impact['waste_hw']
impact['week']     = ['Week 1']*7+['Week 2']*7+['Week 3']*7+['Week 4']*7+['Partial']*2

weekly = (impact[impact['week']!='Partial']
          .groupby('week')[['waste_sq','waste_hw','savings']].sum())

fig, ax = plt.subplots(figsize=(10, 6))
fig.patch.set_facecolor(KN['bg'])
ax.set_facecolor(KN['bg2'])
for sp in ['top','right','left','bottom']: ax.spines[sp].set_visible(False)

x, bw = np.arange(len(weekly)), 0.35
ax.bar(x - bw/2, weekly['waste_sq'], bw, color=KN['alert'],   label='Status Quo')
ax.bar(x + bw/2, weekly['waste_hw'], bw, color=KN['primary'], label='Holt-Winters')

for i, (_, row) in enumerate(weekly.iterrows()):
    ax.text(x[i]+bw/2, row['waste_hw']+impact['waste_sq'].max()*0.02,
            f'-${row["savings"]:,.0f}', ha='center', va='bottom',
            color=KN['text'], fontsize=9, fontweight='bold', fontfamily='monospace')

ax.set_title('Weekly Spoilage Cost: Status Quo vs. Holt-Winters',
             color=KN['text'], pad=14, loc='left')
ax.set_ylabel('Waste Cost ($)', color=KN['secondary'])
ax.set_xticks(x); ax.set_xticklabels(weekly.index, color=KN['secondary'])
ax.tick_params(colors=KN['secondary'])
ax.grid(axis='y', color=KN['grid'], linewidth=0.4)
ax.legend(labelcolor=KN['secondary'])
plt.tight_layout()
plt.savefig('outputs/charts/08_weekly_savings.png', dpi=150)
plt.show()

# Business report numbers
total_sq    = impact['waste_sq'].sum()
total_hw    = impact['waste_hw'].sum()
savings_30d = total_sq - total_hw
pct_saved   = savings_30d / total_sq * 100
N_STORES    = 54

print('=' * 60)
print('HOLT-WINTERS FINANCIAL IMPACT — 30-DAY PILOT')
print('=' * 60)
print(f'  Status quo waste:          ${total_sq:>10,.2f}')
print(f'  Holt-Winters waste:        ${total_hw:>10,.2f}')
print(f'  Monthly savings (1 store): ${savings_30d:>10,.2f}  (-{pct_saved:.1f}%)')
print('-' * 60)
print(f'  Annualized · 1 store:      ${savings_30d*12:>10,.2f}')
print(f'  Annualized · {N_STORES} stores:    ${savings_30d*12*N_STORES:>10,.2f}')
print('=' * 60)
print()
print('Monday morning decision rule:')
print('  Pull your top 10 perishable SKUs.')
print('  If StdDev(orders) > StdDev(sales), you have an ordering problem,')
print('  not a demand problem. This project shows what to do next.')